# 2. Data Preprocessing - Works on Any Dataset
**Steps:** Pick file -> Clean basics -> Missing values -> Outliers -> Encoding -> Scaling -> Train/Test split -> Save

> Run cells top to bottom. A **file explorer window** opens in Step 2 and you will be asked for the **target column** (press Enter to skip if there is none).

## Step 0: Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (LabelEncoder, StandardScaler, MinMaxScaler,
                                   RobustScaler, OneHotEncoder)
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

## Step 1: CONFIG (change only if needed, defaults are fine)

In [ ]:
NUM_IMPUTE      = 'median'    # 'mean' | 'median' | 'knn'      -> how to fill missing numeric values
OUTLIER_DETECT  = 'iqr'       # 'iqr' | 'zscore'
OUTLIER_METHOD  = 'cap'       # 'cap' (winsorize) | 'remove' | 'none'
SCALER_NAME     = 'standard'  # 'standard' | 'minmax' | 'robust'
ORDINAL_MAP     = {}          # ordered categories, low -> high. e.g. {'size': ['S', 'M', 'L']}
DROP_COLS       = []          # columns to drop manually, e.g. ['id', 'name']
TEST_SIZE       = 0.2
RANDOM_STATE    = 42

## Step 2: Select Dataset from File Explorer

In [ ]:
def pick_file():
    """Opens the OS file explorer. Falls back to typing the path if no GUI is available."""
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)      # bring dialog to front
        path = filedialog.askopenfilename(
            title='Select your dataset',
            filetypes=[('Data files', '*.csv *.xlsx *.xls *.json *.txt *.tsv'), ('All files', '*.*')])
        root.destroy()
    except Exception as e:
        print('File dialog not available -> type the path instead.')
        path = input('Enter full file path: ')
    return path.strip().strip('"').strip("'")

def load_data(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.xlsx', '.xls'):
        return pd.read_excel(path)
    if ext == '.json':
        return pd.read_json(path)
    try:                                       # csv / txt / tsv (delimiter auto-detected)
        return pd.read_csv(path, sep=None, engine='python')
    except UnicodeDecodeError:
        return pd.read_csv(path, sep=None, engine='python', encoding='latin-1')

file_path = pick_file()
if not file_path:
    raise SystemExit('No file selected. Re-run this cell.')

df = load_data(file_path)
print('Loaded:', file_path)
print('Shape :', df.shape)
df.head()

In [ ]:
# Choose target column (optional)
df.columns = df.columns.astype(str).str.strip()
print('Columns:', list(df.columns))
TARGET = input('Enter target column name (press Enter to skip): ').strip()
if TARGET == '':
    TARGET = None
elif TARGET not in df.columns:
    print(f"'{TARGET}' not found -> continuing without a target")
    TARGET = None
print('TARGET =', TARGET)

## Step 3: Basic Cleaning

In [ ]:
print('Data types:'); print(df.dtypes, '\n')
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# 1) Strip spaces in text columns, 2) turn placeholder strings into real NaN
def text_cols(d):
    return [col for col in d.columns
            if pd.api.types.is_object_dtype(d[col]) or pd.api.types.is_string_dtype(d[col])]

for col in text_cols(df):
    df[col] = df[col].where(df[col].isna(), df[col].astype(str).str.strip())
placeholders = ['?', 'NA', 'N/A', 'na', 'n/a', 'NaN', 'nan', 'null', 'NULL', 'None', '-', '--', '']
df = df.replace(placeholders, np.nan)

# 3) Text columns that are really numbers (e.g. '12', '3.5') -> convert to numeric
for col in text_cols(df):
    conv = pd.to_numeric(df[col], errors='coerce')
    if df[col].notna().sum() > 0 and conv.notna().sum() >= 0.9 * df[col].notna().sum():
        df[col] = conv

# 4) Booleans -> 0/1
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

# 5) Drop rows where TARGET is missing
if TARGET:
    df = df.dropna(subset=[TARGET]).reset_index(drop=True)

print('Shape after basic cleaning:', df.shape)

In [ ]:
# Drop useless columns: manual list, >50% missing, constant, ID-like text, datetime
to_drop = [col for col in DROP_COLS if col in df.columns]
for col in df.columns:
    if col == TARGET:
        continue
    if df[col].isnull().mean() > 0.5:
        to_drop.append(col)
    elif df[col].nunique(dropna=True) <= 1:
        to_drop.append(col)
    elif col in text_cols(df) and df[col].nunique() == len(df):
        to_drop.append(col)
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        to_drop.append(col)
to_drop = list(dict.fromkeys(to_drop))
print('Dropping columns:', to_drop if to_drop else 'None')
df = df.drop(columns=to_drop)

df_clean = df.copy()      # snapshot (used later for the sklearn Pipeline version)
df_clean.shape

## Step 4: Missing Values

In [ ]:
def missing_report(d):
    m = pd.DataFrame({'missing': d.isnull().sum(), 'missing_%': (d.isnull().mean()*100).round(2)})
    return m[m['missing'] > 0]

print('BEFORE imputation:')
rep = missing_report(df)
display(rep) if len(rep) else print('No missing values')

if df.isnull().any().any():
    plt.figure(figsize=(10, 3.5))
    sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
    plt.title('Missing values map (yellow = missing)')
    plt.show()

In [ ]:
num_feats = [col for col in df.select_dtypes(include=np.number).columns if col != TARGET]
cat_feats = [col for col in df.select_dtypes(exclude=np.number).columns if col != TARGET]

# Numeric: mean / median / KNN
if num_feats and df[num_feats].isnull().any().any():
    if NUM_IMPUTE == 'knn':
        df[num_feats] = KNNImputer(n_neighbors=5).fit_transform(df[num_feats])
    else:
        df[num_feats] = SimpleImputer(strategy=NUM_IMPUTE).fit_transform(df[num_feats])

# Categorical: most frequent value (mode)
if cat_feats and df[cat_feats].isnull().any().any():
    df[cat_feats] = SimpleImputer(strategy='most_frequent').fit_transform(df[cat_feats])

print(f'AFTER imputation (numeric = {NUM_IMPUTE}, categorical = mode): total missing =', df.isnull().sum().sum())

## Step 5: Outliers

In [ ]:
# Only continuous numeric columns (skip target, binary and low-cardinality columns)
out_cols = [col for col in num_feats if df[col].nunique() > 10]

def outlier_bounds(s, method):
    if method == 'zscore':
        m, sd = s.mean(), s.std()
        return m - 3*sd, m + 3*sd
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr

rows = []
for col in out_cols:
    lo, hi = outlier_bounds(df[col], OUTLIER_DETECT)
    n = ((df[col] < lo) | (df[col] > hi)).sum()
    rows.append([col, round(lo, 3), round(hi, 3), n, round(100*n/len(df), 2)])
outlier_table = pd.DataFrame(rows, columns=['column', 'lower_bound', 'upper_bound', 'n_outliers', '%'])
print(f'Outlier detection method: {OUTLIER_DETECT.upper()}')
outlier_table

In [ ]:
df_before_outlier = df.copy()
print('Shape before:', df.shape, '| treatment =', OUTLIER_METHOD)

if OUTLIER_METHOD == 'cap':                       # replace extreme values by the bounds
    for col in out_cols:
        lo, hi = outlier_bounds(df[col], OUTLIER_DETECT)
        df[col] = df[col].clip(lo, hi)
elif OUTLIER_METHOD == 'remove':                  # drop rows containing any outlier
    mask = pd.Series(False, index=df.index)
    for col in out_cols:
        lo, hi = outlier_bounds(df[col], OUTLIER_DETECT)
        mask |= (df[col] < lo) | (df[col] > hi)
    df = df[~mask].reset_index(drop=True)

print('Shape after :', df.shape)

In [ ]:
# Before vs After boxplots (first 4 continuous columns)
show = out_cols[:4]
if show and OUTLIER_METHOD != 'none':
    fig, axes = plt.subplots(2, len(show), figsize=(4*len(show), 7), squeeze=False)
    for i, col in enumerate(show):
        sns.boxplot(y=df_before_outlier[col], ax=axes[0, i], color='lightcoral')
        axes[0, i].set_title(f'{col} (before)')
        sns.boxplot(y=df[col], ax=axes[1, i], color='lightgreen')
        axes[1, i].set_title(f'{col} (after)')
    plt.tight_layout()
    plt.show()

## Step 6: Encoding Categorical Variables

In [ ]:
# (a) Target: Label Encoding (text classes -> 0,1,2,...)
if TARGET and not pd.api.types.is_numeric_dtype(df[TARGET]):
    le = LabelEncoder()
    df[TARGET] = le.fit_transform(df[TARGET].astype(str))
    print('Target label mapping:', {cls: int(i) for i, cls in enumerate(le.classes_)})

# (b) Ordinal Encoding: columns with a natural order (set ORDINAL_MAP in CONFIG)
for col, order in ORDINAL_MAP.items():
    if col in df.columns:
        df[col] = df[col].map({v: i for i, v in enumerate(order)})
        print(f'Ordinal encoded {col}:', {v: i for i, v in enumerate(order)})

# (c) Remaining text columns
cat_feats = [col for col in df.select_dtypes(exclude=np.number).columns if col != TARGET]
high_card = [col for col in cat_feats if df[col].nunique() > 15]    # too many categories
ohe_cols  = [col for col in cat_feats if col not in high_card]

# Frequency encoding for high-cardinality columns
for col in high_card:
    df[col] = df[col].map(df[col].value_counts(normalize=True))
print('Frequency encoded (high cardinality):', high_card if high_card else 'None')

# One-Hot Encoding for the rest (drop_first avoids the dummy variable trap)
print('One-hot encoded:', ohe_cols if ohe_cols else 'None')
df_encoded = pd.get_dummies(df, columns=ohe_cols, drop_first=True, dtype=int)

print('Shape after encoding:', df_encoded.shape)
df_encoded.head()

## Step 7: Feature Scaling

In [ ]:
SCALERS = {'standard': StandardScaler(), 'minmax': MinMaxScaler(), 'robust': RobustScaler()}

feature_cols = [col for col in df_encoded.columns if col != TARGET]
scale_cols   = [col for col in feature_cols if df_encoded[col].nunique() > 2]   # skip 0/1 dummy columns
print('Columns to scale:', scale_cols)

# Compare the three scalers on all scalable columns
if scale_cols:
    for name, sc in SCALERS.items():
        scaled = pd.DataFrame(sc.fit_transform(df_encoded[scale_cols]), columns=scale_cols)
        print(f'\n--- {name.upper()} ---')
        display(scaled.describe().loc[['mean', 'std', 'min', 'max']].round(3))

In [ ]:
# Visual comparison on the first scalable column
if scale_cols:
    col = scale_cols[0]
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
    sns.histplot(df_encoded[col], kde=True, ax=axes[0], color='gray');  axes[0].set_title(f'{col} (original)')
    for ax, (name, sc) in zip(axes[1:], SCALERS.items()):
        vals = sc.fit_transform(df_encoded[[col]]).ravel()
        sns.histplot(vals, kde=True, ax=ax);  ax.set_title(f'{name}')
    plt.tight_layout()
    plt.show()

## Step 8: Train/Test Split + Scaling (correct way - no data leakage)
The scaler is **fitted on the training set only**, then applied to the test set.

In [ ]:
scaler = SCALERS[SCALER_NAME]

if TARGET:
    X = df_encoded.drop(columns=TARGET)
    y = df_encoded[TARGET]
    is_clf = y.nunique() <= 10
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y if is_clf else None)
    except ValueError:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    X_train, X_test = X_train.copy(), X_test.copy()
    if scale_cols:
        X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])   # fit ONLY on train
        X_test[scale_cols]  = scaler.transform(X_test[scale_cols])        # just transform test
    print('X_train:', X_train.shape, '| X_test:', X_test.shape)
    print('Task looks like:', 'classification' if is_clf else 'regression')
    display(X_train.head())
else:
    X_scaled = df_encoded.copy()
    if scale_cols:
        X_scaled[scale_cols] = scaler.fit_transform(X_scaled[scale_cols])
    print('No target given -> full dataset scaled:', X_scaled.shape)
    display(X_scaled.head())

## Step 9 (Alternative): Whole Preprocessing as one sklearn Pipeline / ColumnTransformer

In [ ]:
# Same job in a few lines (imputation + one-hot + scaling). Good to show in the exam.
if TARGET:
    X_raw = df_clean.drop(columns=TARGET)
    y_raw = df_clean[TARGET]
    if not pd.api.types.is_numeric_dtype(y_raw):
        y_raw = pd.Series(LabelEncoder().fit_transform(y_raw.astype(str)), index=y_raw.index)

    num_c = X_raw.select_dtypes(include=np.number).columns.tolist()
    cat_c = X_raw.select_dtypes(exclude=np.number).columns.tolist()

    num_pipe = Pipeline([('impute', SimpleImputer(strategy='median' if NUM_IMPUTE == 'knn' else NUM_IMPUTE)),
                         ('scale',  SCALERS[SCALER_NAME])])
    cat_pipe = Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                         ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

    preprocessor = ColumnTransformer([('num', num_pipe, num_c),
                                      ('cat', cat_pipe, cat_c)])

    Xtr, Xte, ytr, yte = train_test_split(X_raw, y_raw, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    Xtr_p = preprocessor.fit_transform(Xtr)       # fit on train
    Xte_p = preprocessor.transform(Xte)           # transform test

    print('Processed train shape:', Xtr_p.shape, '| test shape:', Xte_p.shape)
    pd.DataFrame(Xtr_p, columns=preprocessor.get_feature_names_out()).head()
else:
    print('Set a TARGET to run the pipeline version')

## Step 10: Save the Preprocessed Data

In [ ]:
df_encoded.to_csv('preprocessed_data_unscaled.csv', index=False)
print('Saved in:', os.getcwd())
print(' - preprocessed_data_unscaled.csv  (cleaned + encoded, not scaled)')

if TARGET:
    pd.concat([X_train, y_train], axis=1).to_csv('train_preprocessed.csv', index=False)
    pd.concat([X_test,  y_test],  axis=1).to_csv('test_preprocessed.csv',  index=False)
    print(' - train_preprocessed.csv / test_preprocessed.csv  (scaled)')

## Quick Theory Cheat-Sheet (for viva / written answers)

| Task | Options | When to use |
|---|---|---|
| **Missing (numeric)** | Mean / Median / KNN | Mean if symmetric, **median if skewed or has outliers**, KNN if columns are related |
| **Missing (categorical)** | Mode | Fill with most frequent category |
| **Drop columns/rows** | >50% missing column; missing target row | Too little information to impute reliably |
| **Outliers (IQR)** | Q1 - 1.5 IQR, Q3 + 1.5 IQR | Works for any shape; robust |
| **Outliers (Z-score)** | \|z\| > 3 | Assumes approx. normal data |
| **Cap vs Remove** | Cap (winsorize) / Remove | Cap keeps all rows (small data); remove if outliers are errors |
| **Label encoding** | 0,1,2,.. | Target variable, or ordinal features |
| **One-hot encoding** | dummy columns (`drop_first=True`) | Nominal features (no order) |
| **Frequency encoding** | category -> its frequency | Very many categories |
| **StandardScaler** | (x - mean) / std | Default choice; SVM, KNN, regression, PCA |
| **MinMaxScaler** | (x - min) / (max - min) -> [0,1] | Neural networks, bounded range needed |
| **RobustScaler** | (x - median) / IQR | Data with outliers |

**Key point:** always fit imputers/scalers on the **training set only**, then transform the test set - otherwise information leaks from test to train (**data leakage**). Tree-based models (Decision Tree, Random Forest) do not need scaling.